In [ ]:
! pip install torch

In [ ]:
! pip install torchvision

In [6]:
! pip install unsloth

  Using cached unsloth-2026.4.4-py3-none-any.whl.metadata (55 kB)
  Using cached unsloth_zoo-2026.4.6-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.10.0-3-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached tyro-1.0.13-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached sentencepiece-0.2.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.19.0-py3-none-any.whl.metadata (15 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached diffusers-0.37.1-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)


In [7]:
! pip install albumentations

  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached stringzilla-4.6.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_28_x86_64.whl.metadata (121 kB)
  Using cached simsimd-6.5.16-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (70 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached albumentations-2.0.8-py3-none-any.whl (369 kB)
Using cached albucore-0.0.24-py3-none-any.whl (15 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (60.4 MB)
Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
Using cached simsimd-6.5.16-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28

In [8]:
! pip install transformers

Managing memory more flexibly to avoid fragmentation

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

 Loading images even if they're slightly corrupted instead of crashing

In [2]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

### Installation

Detecting the PyTorch version and sets the matching xformers version to install

In [1]:
import re
import torch
v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
print(f"PyTorch: {v} → installing {xformers}")

PyTorch: 2.12 → installing xformers==0.0.29.post3


Installing the needed dependencies

In [2]:
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

### Unsloth

Let's prepare the OCR model to our local first

Download DeepSeek-OCR files from haggingface

In [3]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR", local_dir = "deepseek_ocr")

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

'/home/jovyan/deepseek_ocr'

Loads the DeepSeek-OCR model with Unsloth's optimizations

Loading all image/transcription (RIMES + collected) for train/test

In [4]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.   
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
    max_seq_length = 4096,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0521 09:14:43.857000 2152 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.11: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 9.0. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/opt/conda/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import os, random
random.seed(42)

collected_2016 = "data/Coll/2016"
collected_2021 = "data/Coll/2021"
rimes_pages    = "data/Rimes"

images_path_2016  = os.path.join(collected_2016, "images")
texts_path_2016   = os.path.join(collected_2016, "transcriptions")
images_path_2021  = os.path.join(collected_2021, "images")
texts_path_2021   = os.path.join(collected_2021, "transcriptions")
images_path_rimes = os.path.join(rimes_pages, "images")
texts_path_rimes  = os.path.join(rimes_pages, "transcriptions")

images_2016  = sorted([os.path.join(images_path_2016,  f) for f in os.listdir(images_path_2016)])
texts_2016   = sorted([os.path.join(texts_path_2016,   f) for f in os.listdir(texts_path_2016)])
images_2021  = sorted([os.path.join(images_path_2021,  f) for f in os.listdir(images_path_2021)])
texts_2021   = sorted([os.path.join(texts_path_2021,   f) for f in os.listdir(texts_path_2021)])
images_rimes = sorted([os.path.join(images_path_rimes, f) for f in os.listdir(images_path_rimes)])
texts_rimes  = sorted([os.path.join(texts_path_rimes,  f) for f in os.listdir(texts_path_rimes)])

# --- RIMES: 200 test, 600 train ---
all_rimes_idx  = list(range(len(images_rimes)))
idx_rimes_test = random.sample(all_rimes_idx, 160)
idx_rimes_train = [i for i in all_rimes_idx if i not in idx_rimes_test]

# --- Collected: 25 test, rest train ---
idx_2016_test = random.sample(range(len(images_2016)), 18)
idx_2021_test = random.sample(range(len(images_2021)), 18)

# --- Test sets ---
test_images = [images_rimes[i] for i in idx_rimes_test] + \
              [images_2016[i]  for i in idx_2016_test]  + \
              [images_2021[i]  for i in idx_2021_test]
test_texts  = [texts_rimes[i]  for i in idx_rimes_test] + \
              [texts_2016[i]   for i in idx_2016_test]  + \
              [texts_2021[i]   for i in idx_2021_test]

# --- Train sets ---
train_images_rimes = [images_rimes[i] for i in idx_rimes_train]
train_texts_rimes  = [texts_rimes[i]  for i in idx_rimes_train]
train_images_2016  = [p for i,p in enumerate(images_2016) if i not in idx_2016_test]
train_texts_2016   = [p for i,p in enumerate(texts_2016)  if i not in idx_2016_test]
train_images_2021  = [p for i,p in enumerate(images_2021) if i not in idx_2021_test]
train_texts_2021   = [p for i,p in enumerate(texts_2021)  if i not in idx_2021_test]

# --- Oversample collected 3x ---
images = train_images_rimes + (train_images_2016 + train_images_2021)
texts  = train_texts_rimes  + (train_texts_2016  + train_texts_2021)

print(f"Train : {len(images)} ({len(train_images_rimes)} RIMES + {len(train_images_2016)+len(train_images_2021)} collected)")
print(f"Test  : {len(test_images)} ({len(idx_rimes_test)} RIMES + {len(idx_2016_test)+len(idx_2021_test)} collected)")

Train : 791 (648 RIMES + 143 collected)
Test  : 196 (160 RIMES + 36 collected)


Copy the transcriptions to the working space

In [6]:
import os, shutil

transcriptions_dir = "data_aug/all/transcriptions"
os.makedirs(transcriptions_dir, exist_ok=True)

seen_paths = set()
unique_txt_paths = []
for path in texts + test_texts:
    if path not in seen_paths:
        seen_paths.add(path)
        unique_txt_paths.append(path)

for txt_path in unique_txt_paths:
    shutil.copy(txt_path, os.path.join(transcriptions_dir, os.path.basename(txt_path)))

print(f"Copied {len(unique_txt_paths)} transcription files → {transcriptions_dir}")

def to_working_path(p):
    return os.path.join(transcriptions_dir, os.path.basename(p))

texts      = [to_working_path(p) for p in texts]
test_texts = [to_working_path(p) for p in test_texts]

print("Text paths updated.")

Copied 986 transcription files → data_aug/all/transcriptions
Text paths updated.


Convert images to JPG and copy them to the working space

In [7]:
from PIL import Image, ImageFile
from pathlib import Path
ImageFile.LOAD_TRUNCATED_IMAGES = True

output_folder = "data_aug/all/images"
os.makedirs(output_folder, exist_ok=True)
corrupted = []

def convert_to_jpg(src_path, dst_path):
    try:
        with Image.open(src_path) as im:
            im.convert('RGB').save(dst_path, quality=95)
    except Exception as e:
        corrupted.append((src_path, str(e)))

all_img_paths = list(set(images + test_images))   # ← removed eval_images
for img_path in all_img_paths:
    out = Path(output_folder) / f"{Path(img_path).stem}.jpg"
    convert_to_jpg(img_path, out)

images      = [str(Path(output_folder) / f"{Path(p).stem}.jpg") for p in images]
test_images = [str(Path(output_folder) / f"{Path(p).stem}.jpg") for p in test_images]  # ← removed eval_images

print(f"Converted {len(all_img_paths)} images")
print(f"Corrupted ({len(corrupted)}):")
for path, error in corrupted:
    print(f"  {path} → {error}")

print(f"\nTrain  : {len(images)}")
print(f"Test   : {len(test_images)}")

missing = [(p,'image') for p in images      if not os.path.exists(p)] + \
          [(p,'text')  for p in texts        if not os.path.exists(p)] + \
          [(p,'image') for p in test_images  if not os.path.exists(p)] + \
          [(p,'text')  for p in test_texts   if not os.path.exists(p)]   # ← removed eval lines
print(f"Missing files: {len(missing)}")

Converted 987 images
Corrupted (1):
  data/Coll/2016/images/.ipynb_checkpoints → [Errno 21] Is a directory: 'data/Coll/2016/images/.ipynb_checkpoints'

Train  : 791
Test   : 196
Missing files: 1


Data Augmentation (collected data only)

In [8]:
# === DATA AUGMENTATION FOR COLLECTED DATA ===
# It creates augmented copies of collected images only (not RIMES).

import albumentations as A
import cv2, os, numpy as np
from pathlib import Path

aug = A.Compose([
    A.OneOf([
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.MotionBlur(blur_limit=3, p=0.3),
    ], p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.3),
    A.Affine(scale=(0.95, 1.05), rotate=(-2, 2), shear=(-2, 2),
             border_mode=cv2.BORDER_CONSTANT, value=(255, 255, 255), p=0.4),
    A.Perspective(scale=(0.02, 0.05), pad_mode=cv2.BORDER_CONSTANT,
                  pad_val=(255, 255, 255), p=0.2),
])

aug_dir = 'data_aug/images_aug'
os.makedirs(aug_dir, exist_ok=True)

# Identify collected (non-RIMES) train images/texts
collected_train_imgs = train_images_2016 + train_images_2021
collected_train_txts = train_texts_2016  + train_texts_2021
# Convert to working paths
collected_train_imgs = [str(Path('data_aug/all/images') / f'{Path(p).stem}.jpg') for p in collected_train_imgs]
collected_train_txts = [str(Path('data_aug/all/transcriptions') / Path(p).name) for p in collected_train_txts]

aug_images, aug_texts = [], []
N_AUG = 2  # number of augmented copies per image
for img_path, txt_path in zip(collected_train_imgs, collected_train_txts):
    img = cv2.imread(img_path)
    if img is None: continue
    for k in range(N_AUG):
        aug_img = aug(image=img)['image']
        out_name = f'{Path(img_path).stem}_aug{k}.jpg'
        out_path = os.path.join(aug_dir, out_name)
        cv2.imwrite(out_path, aug_img)
        aug_images.append(out_path)
        aug_texts.append(txt_path)  # same transcription

# Append augmented data to training set
images += aug_images
texts  += aug_texts
print(f'Added {len(aug_images)} augmented collected images. New train size: {len(images)}')

ModuleNotFoundError: No module named 'albumentations'

In [ ]:
print(f"Train images : {len(images)}")
print(f"Train texts  : {len(texts)}")
print(f"Test  images : {len(test_images)}")
print(f"Test  texts  : {len(test_texts)}")

print(f"\nSample train pair:\n  {images[0]}\n  {texts[0]}")
print(f"\nSample test pair:\n  {test_images[0]}\n  {test_texts[0]}")

missing = [(p,'image') for p in images       if not os.path.exists(p)] + \
          [(p,'text')  for p in texts        if not os.path.exists(p)] + \
          [(p,'image') for p in test_images  if not os.path.exists(p)] + \
          [(p,'text')  for p in test_texts   if not os.path.exists(p)]

print(f"\nMissing files: {len(missing)}")
for path, kind in missing:
    print(f"  [{kind}] {path}")

Testing some prompts

In [ ]:
'''prompt = "<image>\n<|grounding|>Free OCR French."
res = model.infer(
    tokenizer, prompt=prompt, image_file=image_file,
    output_path='/kaggle/working/',
    base_size=1024, image_size=640,
    crop_mode=True, save_results=False, test_compress=False
)'''

In [ ]:
'''prompt = "<image>\n<|grounding|>Free OCR."
res = model.infer(
    tokenizer, prompt=prompt, image_file=image_file,
    output_path='/kaggle/working/',
    base_size=1024, image_size=640,
    crop_mode=True, save_results=False, test_compress=False
)'''

In [ ]:
'''prompt = "<image>\n<|grounding|>Free OCR French Handwritten Medical Notes."
res = model.infer(
    tokenizer, prompt=prompt, image_file=image_file,
    output_path='/kaggle/working/',
    base_size=1024, image_size=640,
    crop_mode=True, save_results=False, test_compress=False
)'''

In [ ]:
'''prompt = "<image>\n<|grounding|>FREE OCR. Handwritten French document. It may be a letter, email, medical note, or administrative document."
res = model.infer(
    tokenizer, prompt=prompt, image_file=image_file,
    output_path='/kaggle/working/',
    base_size=1024, image_size=640,
    crop_mode=True, save_results=False, test_compress=False
)'''

In [ ]:
'''prompt = "<image>\n<|grounding|>Transcribe this handwritten French document exactly as written. It may be a letter, email, medical note, or administrative document. Preserve all line breaks, punctuation, accents, and original content without interpretation or correction."
res = model.infer(
    tokenizer, prompt=prompt, image_file=image_file,
    output_path='/kaggle/working/',
    base_size=1024, image_size=640,
    crop_mode=True, save_results=False, test_compress=False
)'''

In [ ]:
'''prompt = "<image> <|grounding|> Transcrivez intégralement le texte de ce document manuscrit français avec une précision maximale. Structurez la sortie pour préserver scrupuleusement : 1. **La mise en page originale** (sauts de ligne, paragraphes, alinéas, tabulations). 2. **Tous les signes diacritiques** (accents aigus, graves, circonflexes, cédilles, trémas). 3. **L'orthographe d'origine** (y compris les formes historiques, archaïques ou les fautes d'époque). 4. **Le contenu numérique** (dates, montants, quantités). 5. **Les ajouts marginaux** (annotations, notes en marge, renvois). "
image_file = images[170]
res = model.infer(
    tokenizer, prompt=prompt, image_file=image_file,
    output_path='/kaggle/working/',
    base_size=1024, image_size=640,
    crop_mode=True, save_results=False, test_compress=False
)'''

# Let's finetune Deepseek-OCR !

Adding LoRA to the model

In [ ]:
# LoRA
model = FastVisionModel.get_peft_model(
    model,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep


To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "<|User|>",
  "content": "",
  "images": []
},
{ "role": "<|Assistant|>",
  "content": ""
},
]
```

In [ ]:
instruction = "<image>\n<|grounding|>Free OCR French."
from PIL import Image
def convert_to_conversation(img, txt):
    """Convert dataset sample to conversation format"""
    with open(txt, 'r', encoding='utf-8') as file:
        content = file.read()
    image = Image.open(img)
    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [image]
        },
        {
            "role": "<|Assistant|>",
            "content": content
        },
    ]
    return {"messages": conversation}

Let's convert the training dataset into the "correct" format for finetuning:

In [ ]:
converted_dataset      = [convert_to_conversation(im, txt) for im, txt in zip(images,      texts)]
print(f"Train: {len(converted_dataset)}")

We look at how the conversations are structured for the first example:

In [ ]:
converted_dataset[0]

Create datacollator

In [ ]:
# @title Create datacollator

import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr.modeling_deepseekocr import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCRDataCollator:
    """
    Args:
        tokenizer: Tokenizer
        model: Model
        image_size: Size for image patches (default: 640)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses (mask user prompts)
    """
    tokenizer: Any
    model: Any
    image_size: int = 520   # was 640
    base_size: int = 640   # was 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 520,   # was 640
        base_size: int = 640,    # was 1024
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype  # Get dtype from model
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # Get BOS token ID from tokenizer
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (bytes dict or PIL Image) to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num + 1) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds

        Returns:
            Tuple of (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # Determine crop ratio based on image size
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=9,
                    image_size=self.image_size, use_thumbnail=False
                )

            # Process global view with padding
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # Process local views (crops) if applicable
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            # Calculate image tokens
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base + [self.image_token_id]) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num) + [self.image_token_id]) * (
                    num_queries * height_crop_num)

        else:  # crop_mode = False
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            # For smaller base sizes, resize; for larger, pad
            if self.base_size <= 640:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color=tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries + [self.image_token_id]) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
            """
            Process a single conversation into model inputs.
            """

            # --- 1. Setup ---
            images = []
            for message in messages:
                if "images" in message and message["images"]:
                    for img_data in message["images"]:
                        if img_data is not None:
                            pil_image = self.deserialize_image(img_data)
                            images.append(pil_image)

            if not images:
                raise ValueError("No images found in sample. Please ensure all samples contain images.")

            tokenized_str = []
            images_seq_mask = []
            images_list, images_crop_list, images_spatial_crop = [], [], []

            prompt_token_count = -1 # Index to start training
            assistant_started = False
            image_idx = 0

            # Add BOS token at the very beginning
            tokenized_str.append(self.bos_id)
            images_seq_mask.append(False)

            for message in messages:
                role = message["role"]
                content = message["content"]

                # Check if this is the assistant's turn
                if role == "<|Assistant|>":
                    if not assistant_started:
                        # This is the split point. All tokens added *so far*
                        # are part of the prompt.
                        prompt_token_count = len(tokenized_str)
                        assistant_started = True

                    # Append the EOS token string to the *end* of assistant content
                    content = f"{content.strip()} {self.tokenizer.eos_token}"

                # Split this message's content by the image token
                text_splits = content.split('<image>')

                for i, text_sep in enumerate(text_splits):
                    # Tokenize the text part
                    tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                    tokenized_str.extend(tokenized_sep)
                    images_seq_mask.extend([False] * len(tokenized_sep))

                    # If this text is followed by an <image> tag
                    if i < len(text_splits) - 1:
                        if image_idx >= len(images):
                            raise ValueError(
                                f"Data mismatch: Found '<image>' token but no corresponding image."
                            )

                        # Process the image
                        image = images[image_idx]
                        img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                        images_list.extend(img_list)
                        images_crop_list.extend(crop_list)
                        images_spatial_crop.extend(spatial_crop)

                        # Add image placeholder tokens
                        tokenized_str.extend(tok_img)
                        images_seq_mask.extend([True] * len(tok_img))

                        image_idx += 1 # Move to the next image

            # --- 3. Validation and Final Prep ---
            if image_idx != len(images):
                raise ValueError(
                    f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
                )

            # If we never found an assistant message, we're in a weird state
            # (e.g., user-only prompt). We mask everything.
            if not assistant_started:
                print("Warning: No assistant message found in sample. Masking all tokens.")
                prompt_token_count = len(tokenized_str)

            # Prepare image tensors
            images_ori = torch.stack(images_list, dim=0)
            images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)

            if images_crop_list:
                images_crop = torch.stack(images_crop_list, dim=0)
            else:
                images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)

            return {
                "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
                "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
                "images_ori": images_ori,
                "images_crop": images_crop,
                "images_spatial_crop": images_spatial_crop_tensor,
                "prompt_token_count": prompt_token_count, # This is now accurate
            }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        # Process each sample
        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        # Extract lists
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # Pad sequences
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)

        # Create labels
        labels = input_ids.clone()

        # Mask padding tokens
        labels[labels == self.tokenizer.pad_token_id] = -100

        # Mask image tokens (model shouldn't predict these)
        labels[images_seq_mask] = -100

        # Mask user prompt tokens when train_on_responses_only=True (only train on assistant responses)
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        # Create attention mask
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # Prepare images batch (list of tuples)
        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        # Stack spatial crop info
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

We use our new `DeepSeekOCRDataCollator` which will help in our vision finetuning setup.

In [9]:
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported
FastVisionModel.for_training(model) # Enable for training!
data_collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer,
    model = model,
    image_size= 520,   
    base_size= 640,   
    crop_mode=True,
    train_on_responses_only=True,
)
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator, # Must use!
    train_dataset = converted_dataset, ###############
    args = TrainingArguments(
        per_device_train_batch_size = 2, 
        gradient_accumulation_steps = 4, 
        warmup_steps = 5,
        # max_steps = 15, ###############
        num_train_epochs = 10, ###############
        learning_rate = 5e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        fp16 = not is_bf16_supported(),  # Use fp16 if bf16 is not supported
        bf16 = is_bf16_supported(),  # Use bf16 if supported
        fp16_full_eval = True,
        output_dir = "output",
        report_to = "none",     # For Weights and Biases
        dataloader_num_workers=2,
        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        save_steps = 10,            # ← save every 10 steps  ###############
        save_total_limit = 10,
    ),
)

NameError: name 'DeepSeekOCRDataCollator' is not defined

Show Memory State

In [20]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA H100 NVL MIG 1g.24gb. Max memory = 21.625 GB.
4.195 GB of memory reserved.


Train

In [21]:
import torch
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_cudnn_sdp(False)  # ← disable the problematic backend

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 992 | Num Epochs = 10 | Total steps = 1,240
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 77,509,632 of 3,413,615,872 (2.27% trained)


BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])


Step,Training Loss
1,1.773800
2,1.702100
3,1.760400
4,1.658500
5,1.673700
6,1.619200
7,1.675600
8,1.672700
9,1.528400
10,1.670300


BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


...

BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([6, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1, 100, 1280])
PATCHES:  torch.Size([9, 64, 1280])
BASE:  torch.Size([1

Pick the best step from saved checkpoints and delete the others

In [ ]:
import os

# Get all checkpoint steps
checkpoints = [d for d in os.listdir("output") if d.startswith("checkpoint-")]
saved_steps = sorted([int(c.split("-")[1]) for c in checkpoints])

# Get loss for each saved checkpoint step from trainer log
log_history = trainer.state.log_history
train_logs = {int(l["step"]): l["loss"] for l in log_history if "loss" in l and "eval_loss" not in l}

# Match checkpoints to their losses
checkpoint_losses = {}
for step in saved_steps:
    if step in train_logs:
        checkpoint_losses[step] = train_logs[step]
    else:
        # find nearest logged step
        nearest_log = min(train_logs.keys(), key=lambda x: abs(x - step))
        checkpoint_losses[step] = train_logs[nearest_log]

# Pick best
best_step = min(checkpoint_losses, key=checkpoint_losses.get)
print(f"\nBest checkpoint: checkpoint-{best_step} | loss: {checkpoint_losses[best_step]:.4f}")

In [ ]:
from safetensors.torch import load_file

best_checkpoint = f"output/checkpoint-{best_step}"
weights = load_file(f"{best_checkpoint}/adapter_model.safetensors")
model.load_state_dict(weights, strict=False)
print(f"Loaded weights from {best_checkpoint}")

# Delete ALL checkpoints to free space
shutil.rmtree("output")
print("Checkpoints deleted")

### Test


In [ ]:
import numpy as np
token_counts = []
for txt_path in test_texts:
    with open(txt_path, 'r', encoding='utf-8') as f:
        token_counts.append(len(tokenizer.encode(f.read())))
print(f"Mean: {np.mean(token_counts):.0f} | Median: {np.median(token_counts):.0f} | Max: {max(token_counts)}")

In [ ]:
import jiwer, re, torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

FastVisionModel.for_inference(model)

def extract_text_only(raw):
    # Discard if hallucination loop detected
    if raw.count('text-decoration') > 10:
        print('css/html')
        return ""
    raw = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<\|det\|>.*?<\|/det\|>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<[^>]+>', '', raw)          # remove stray HTML
    raw = re.sub(r'[\w-]+:[\w-]+;', '', raw)   # remove stray CSS
    lines = raw.strip().splitlines()
    seen, deduped = set(), []
    for line in lines:
        line = line.strip()
        if line and line not in seen:
            seen.add(line)
            deduped.append(line)
    return "\n".join(deduped)

# Build collator ONCE — not inside the loop
collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer, model=model,
    image_size=512, base_size=640,
    crop_mode= True, 
    train_on_responses_only=True,
)

def run_inference(img_path):
    image = Image.open(img_path).convert("RGB")
    sample = {"messages": [
        {"role": "<|User|>", "content": "<image>\n<|grounding|>Free OCR French.", "images": [image]},
        {"role": "<|Assistant|>", "content": ""},
    ]}
    processed   = collator.process_single_sample(sample["messages"])
    input_ids   = processed["input_ids"].unsqueeze(0).to(model.device)
    images_seq  = processed["images_seq_mask"].unsqueeze(0).to(model.device)
    images_ori  = processed["images_ori"].unsqueeze(0).to(model.device)
    images_crop = processed["images_crop"].unsqueeze(0).to(model.device)
    images_spat = processed["images_spatial_crop"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            images=[(images_crop.squeeze(0), images_ori.squeeze(0))],
            images_seq_mask=images_seq,
            images_spatial_crop=images_spat,
            max_new_tokens= 1024,   # hard cap — prevents infinite loops ###############
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)

def run_inference_batch(img_paths, max_new_tokens=512):
    """Run inference on a list of images. Returns list of raw predictions."""
    preds = []
    for idx, img_path in enumerate(img_paths):
        try:
            pred = run_inference(img_path)
        except Exception as e:
            print(f'  Inference error on {img_path}: {e}')
            pred = ''
        preds.append(pred)
        if (idx + 1) % 25 == 0:
            print(f'  [{idx+1}/{len(img_paths)}] done...')
    return preds

In [ ]:
'''def word_precision(reference, hypothesis):
    ref_words  = reference.split()
    hyp_words  = hypothesis.split()
    if not hyp_words:
        return 0.0
    # Count how many predicted words appear in reference
    ref_counter = {}
    for w in ref_words:
        ref_counter[w] = ref_counter.get(w, 0) + 1
    correct = 0
    for w in hyp_words:
        if ref_counter.get(w, 0) > 0:
            correct += 1
            ref_counter[w] -= 1
    return correct / len(hyp_words)

def char_precision(reference, hypothesis):
    if not hypothesis:
        return 0.0
    ref_counter = {}
    for c in reference:
        ref_counter[c] = ref_counter.get(c, 0) + 1
    correct = 0
    for c in hypothesis:
        if ref_counter.get(c, 0) > 0:
            correct += 1
            ref_counter[c] -= 1
    return correct / len(hypothesis)'''

In [ ]:
import jiwer, os, pandas as pd

def evaluate_detailed(img_list, txt_list, tag=''):
    """Run eval with split WER/CER reporting. Uses batched inference."""

    # --- Run all predictions at once ---
    print(f'Running inference on {len(img_list)} images...')
    raw_preds = run_inference_batch(img_list)

    # --- Score them ---
    records = []
    skipped_ref, skipped_hyp, errors = 0, 0, 0
    os.makedirs('test_results', exist_ok=True)

    for img_path, txt_path, raw_pred in zip(img_list, txt_list, raw_preds):
        try:
            with open(txt_path, 'r', encoding='utf-8') as f:
                reference = extract_text_only(f.read())
            hypothesis = extract_text_only(raw_pred)

            pred_path = os.path.join('test_results', os.path.basename(txt_path))
            with open(pred_path, 'w', encoding='utf-8') as f:
                f.write(hypothesis)

            if not reference.strip():
                skipped_ref += 1; continue
            if not hypothesis.strip():
                skipped_hyp += 1; continue

            wer = jiwer.wer(reference, hypothesis)
            cer = jiwer.cer(reference, hypothesis)
            exact = 1.0 if reference.strip() == hypothesis.strip() else 0.0
            is_rimes = os.path.basename(img_path).startswith('page_')
            source = 'RIMES' if is_rimes else 'Collected'
            fname = os.path.basename(img_path)
            records.append({'source': source, 'file': fname,
                            'wer': wer, 'cer': cer, 'exact': exact})

        except Exception as e:
            print(f'Error scoring {img_path}: {e}')
            errors += 1

    print(f'Skipped — empty ref: {skipped_ref} | empty hyp: {skipped_hyp} | errors: {errors}')

    if not records:
        print('No valid results.'); return None

    df = pd.DataFrame(records)

    # --- Build summary table ---
    rows = []
    for src in ['RIMES', 'Collected', 'Overall']:
        sub = df if src == 'Overall' else df[df['source'] == src]
        if len(sub) == 0: continue
        ok  = sub[sub['wer'] < 1.0]
        bad = sub[sub['wer'] >= 1.0]
        best_row  = sub.loc[sub['wer'].idxmin()]
        worst_row = sub.loc[sub['wer'].idxmax()]
        rows.append({
            'Source': src,
            'N': len(sub),
            'Exact Match %': f"{sub['exact'].mean()*100:.1f}",
            'Readable (WER<100%)': len(ok),
            'Mean WER (readable)':  f"{ok['wer'].mean():.3f}" if len(ok) else '-',
            'Mean CER (readable)':  f"{ok['cer'].mean():.3f}" if len(ok) else '-',
            'Problematic (WER>=100%)': len(bad),
            'Max WER (problematic)':   f"{bad['wer'].max():.2f}" if len(bad) else '-',
            'Best (file | WER)':  f"{best_row['file']} | {best_row['wer']:.3f}",
            'Worst (file | WER)': f"{worst_row['file']} | {worst_row['wer']:.3f}",
        })

    summary = pd.DataFrame(rows).set_index('Source')
    print(f'\n=== {tag} Results ===')
    print(summary.to_string())
    return df

Evaluation loop

In [ ]:
# --- Run finetuned evaluation ---
FastVisionModel.for_inference(model)
ft_results = evaluate_detailed(test_images, test_texts, tag='FINETUNED')

Before - After finetuning

In [ ]:
# --- Compare baseline vs finetuned ---
if baseline_results is not None and ft_results is not None:
    for src in ['RIMES', 'Collected', 'Overall']:
    bl = baseline_results if src == 'Overall' else baseline_results[baseline_results['source'] == src]
    ft = ft_results       if src == 'Overall' else ft_results[ft_results['source'] == src]
    if len(bl) == 0 or len(ft) == 0: continue
    bl_ok = bl[bl['wer'] < 1.0]
    ft_ok = ft[ft['wer'] < 1.0]
    print(f'\n--- {src} ---')
    print(f"  Exact Match: {bl['exact'].mean()*100:.1f}% -> {ft['exact'].mean()*100:.1f}%")
    if len(bl_ok) and len(ft_ok):
        print(f"  Mean WER (readable): {bl_ok['wer'].mean():.3f} -> {ft_ok['wer'].mean():.3f}")
        print(f"  Mean CER (readable): {bl_ok['cer'].mean():.3f} -> {ft_ok['cer'].mean():.3f}")
    print(f"  Readable count: {len(bl_ok)} -> {len(ft_ok)}")
    print(f"  Problematic count: {len(bl)-len(bl_ok)} -> {len(ft)-len(ft_ok)}")


### Save the model


Check free space and save

In [ ]:
# Check free space
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Free space: {free / 1024**3:.1f} GB")

free_gb = free / 1024**3
if free_gb > 10:
    model.save_pretrained_merged(
        "fine_tuned_deepseek_ocr_merged",
        tokenizer,
        save_method="merged_16bit",
    )
    print("Merged model saved to /kaggle/working/fine_tuned_deepseek_ocr_merged")
else:
    print(f"Not enough disk space ({free_gb:.1f} GB). Skipping merge — proceeding to next cells.")

Load the saved model and test on one example

In [ ]:
import os

if os.path.exists("./fine_tuned_deepseek_ocr_merged"):
    from transformers import AutoModel
    from unsloth import FastVisionModel

    # Load merged model
    model_merged, tokenizer_merged = FastVisionModel.from_pretrained(
        "./fine_tuned_deepseek_ocr_merged",
        load_in_4bit=False,
        auto_model=AutoModel,
        trust_remote_code=True,
    )
    FastVisionModel.for_inference(model_merged)

    # Run inference
    prompt = "<image>\n<|grounding|>Free OCR French."
    image_file = '/kaggle/input/handwritten2text-training-dataset/Handwritten2Text Training Dataset/Images/eval2011-10_000002.jpg'
    output_path = 'output'

    res = model_merged.infer(
        tokenizer_merged,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        save_results=False,
        test_compress=False
    )
else:
    print("Merged model not found, skipping inference test.")